In [108]:
import pandas as pd
import numpy as np

In [110]:
markers_df = pd.read_csv("lifu_markers_1_2back_-1580742246.csv")
calibration_complete =markers_df[markers_df["marker"] == "collecting_baseline"].iloc[-1]
calibration_complete = calibration_complete["LSL_timestamp"]
markers_df_on = markers_df[markers_df["marker"] =="LIFU_ON"]
markers_df

,Time,marker,LSL_timestamp
0,2.167579,collecting_baseline,1.606245e+06
1,2.201134,collecting_baseline,1.606245e+06
2,2.242651,collecting_baseline,1.606245e+06
3,2.274780,collecting_baseline,1.606245e+06
4,2.320597,collecting_baseline,1.606245e+06
...,...,...,...
201,26.918029,LIFU_ON,1.606270e+06
202,31.094722,LIFU_OFF,1.606274e+06
203,34.233252,LIFU_ON,1.606277e+06
204,38.508772,LIFU_OFF,1.606281e+06


In [111]:
eeg_df = pd.read_csv("eeg_LSL_gpype2back_-1580742246.csv")
eeg_df

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,Ch06
0,1.606244e+06,-12412.763672,804.865234,647808.000000,2.117485e+06,506574.687500,538191.500000
1,1.606244e+06,-12218.762695,842.817688,710341.687500,2.050204e+06,490478.750000,538191.500000
2,1.606244e+06,-11832.976562,866.241028,750373.500000,1.969379e+06,471142.750000,538191.500000
3,1.606244e+06,-12008.380859,876.244141,767803.812500,1.877266e+06,449106.156250,538191.500000
4,1.606244e+06,-12440.919922,874.108765,764066.187500,1.777017e+06,425123.187500,425123.187500
...,...,...,...,...,...,...,...
26377,1.606349e+06,-7354.545898,0.934224,0.872775,3.436062e+00,0.267000,0.255543
26378,1.606349e+06,-6063.342773,0.638702,0.407941,3.407291e+00,0.260117,0.255543
26379,1.606349e+06,-5325.269531,0.319189,0.101881,3.357265e+00,0.248149,0.255543
26380,1.606350e+06,0.000000,-0.011799,0.000139,3.289621e+00,0.231967,0.255543


In [112]:
eeg_df = eeg_df.rename(columns = {"Time": "LSL Time", "Ch01": "Raw", "Ch02": "Theta Bandpass", "Ch03": "Power", "Ch04": "Moving Average", "Ch05": "Z-score", "Ch06": "Hold"})
eeg_df

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold
0,1.606244e+06,-12412.763672,804.865234,647808.000000,2.117485e+06,506574.687500,538191.500000
1,1.606244e+06,-12218.762695,842.817688,710341.687500,2.050204e+06,490478.750000,538191.500000
2,1.606244e+06,-11832.976562,866.241028,750373.500000,1.969379e+06,471142.750000,538191.500000
3,1.606244e+06,-12008.380859,876.244141,767803.812500,1.877266e+06,449106.156250,538191.500000
4,1.606244e+06,-12440.919922,874.108765,764066.187500,1.777017e+06,425123.187500,425123.187500
...,...,...,...,...,...,...,...
26377,1.606349e+06,-7354.545898,0.934224,0.872775,3.436062e+00,0.267000,0.255543
26378,1.606349e+06,-6063.342773,0.638702,0.407941,3.407291e+00,0.260117,0.255543
26379,1.606349e+06,-5325.269531,0.319189,0.101881,3.357265e+00,0.248149,0.255543
26380,1.606350e+06,0.000000,-0.011799,0.000139,3.289621e+00,0.231967,0.255543


In [113]:
lifu_on = markers_df[markers_df["marker"]=="LIFU_ON"].copy()
lifu_on_timestamp = lifu_on["LSL_timestamp"]
lifu_on_timestamp = np.array(lifu_on_timestamp)
lifu_on_timestamp

array([1606269.7886553, 1606277.1038785, 1606348.3945189])

In [114]:
target_ts = lifu_on_timestamp

tol = 1e-3  # tolerance for float comparison

df_matches = eeg_df[
    eeg_df["LSL Time"].apply(
        lambda x: np.any(np.abs(target_ts - x) < tol)
    )
]

df_matches

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold
6525,1.606270e+06,-6744.885742,-1.924152,3.702361,23.382854,5.038961,5.373763
6526,1.606270e+06,-5984.592773,-2.157651,4.655457,21.884073,4.680400,5.373763
8354,1.606277e+06,-6026.674316,1.553836,2.414407,9.532751,1.725539,1.725539
26176,1.606348e+06,-6340.194336,-6.389326,40.823479,10.309962,1.911474,1.568178


In [115]:
merged_df = eeg_df

In [116]:
buffer = []
offline_z = []
last_theta_val = None

MU = 2.32
SIGMA = 4.18
MAD_THRESHOLD = 6
THETA_THRESHOLD_Z = 1.5
INITIAL_CUTOFF = 100.0
BUFFER_SIZE = 500
COOLDOWN_TIME = 7
sonication_enabled = 20

# Ch05 = index 4
channel = "Hold"

for i in range(len(merged_df)):

    theta_val = merged_df[channel].iloc[i]   # MUST match online
    if last_theta_val is not None and theta_val == last_theta_val:
        offline_z.append(last_theta_val) 
        continue
    # Warm-up
    if len(buffer) <= 200:
        if theta_val < INITIAL_CUTOFF:
            buffer.append(theta_val)
        offline_z.append(np.nan)
        continue

    # Rolling buffer
    if len(buffer) > BUFFER_SIZE:
        buffer.pop(0)

    arr = np.array(buffer)
    median = np.median(arr)
    mad = np.median(np.abs(arr - median)) + 1e-6

    # Artifact rejection
    z_art = abs(theta_val - median) / mad
    if z_art > MAD_THRESHOLD:
        offline_z.append(np.nan)
        continue

    # Clean sample
    buffer.append(theta_val)
    last_theta_val = theta_val

    # Z-score
    offline_z.append(theta_val)

merged_df["offline z-score"] = offline_z




In [117]:
LIFU = np.zeros(len(merged_df))
last_trigger_time = -999

for i in range(len(merged_df)):
    theta_z = merged_df["offline z-score"].iloc[i]
    now = merged_df["LSL Time"].iloc[i]

    if now> sonication_enabled and theta_z > THETA_THRESHOLD_Z and theta_z < MAD_THRESHOLD and (now - last_trigger_time > COOLDOWN_TIME):
        LIFU[i] = 1.0
        last_trigger_time = now


In [118]:
merged_df["LIFU"] = LIFU
merged_df

,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,offline z-score,LIFU
0,1.606244e+06,-12412.763672,804.865234,647808.000000,2.117485e+06,506574.687500,538191.500000,NaN,0.0
1,1.606244e+06,-12218.762695,842.817688,710341.687500,2.050204e+06,490478.750000,538191.500000,NaN,0.0
2,1.606244e+06,-11832.976562,866.241028,750373.500000,1.969379e+06,471142.750000,538191.500000,NaN,0.0
3,1.606244e+06,-12008.380859,876.244141,767803.812500,1.877266e+06,449106.156250,538191.500000,NaN,0.0
4,1.606244e+06,-12440.919922,874.108765,764066.187500,1.777017e+06,425123.187500,425123.187500,NaN,0.0
...,...,...,...,...,...,...,...,...,...
26377,1.606349e+06,-7354.545898,0.934224,0.872775,3.436062e+00,0.267000,0.255543,0.255543,0.0
26378,1.606349e+06,-6063.342773,0.638702,0.407941,3.407291e+00,0.260117,0.255543,0.255543,0.0
26379,1.606349e+06,-5325.269531,0.319189,0.101881,3.357265e+00,0.248149,0.255543,0.255543,0.0
26380,1.606350e+06,0.000000,-0.011799,0.000139,3.289621e+00,0.231967,0.255543,0.255543,0.0


In [120]:
mask_lifu = merged_df["LIFU"] == 1.0
lifu_on = merged_df[mask_lifu]
lifu_on["LSL Time"] = lifu_on["LSL Time"]
lifu_on_cal = lifu_on[lifu_on["LSL Time"] > calibration_complete]
lifu_on_cal

C:\Users\jshin\AppData\Local\Temp\ipykernel_28520\10077578.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lifu_on["LSL Time"] = lifu_on["LSL Time"]


,LSL Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,offline z-score,LIFU
4424,1.606261e+06,-4672.957031,4.924239,24.248133,10.425403,1.939091,1.939091,1.939091,1.0
6172,1.606268e+06,-5719.280273,-2.243361,5.032667,23.728006,5.121532,4.539615,4.539615,1.0
8354,1.606277e+06,-6026.674316,1.553836,2.414407,9.532751,1.725539,1.725539,1.725539,1.0
26174,1.606348e+06,-6113.863281,-5.905739,34.877747,8.874983,1.568178,1.568178,1.568178,1.0


In [127]:
lifu_on_time = lifu_on_cal["LSL Time"]
lifu_on_time = np.array(lifu_on_time)
lifu_on_final = lifu_on_time[1:]

In [128]:
lifu_on_final - target_ts

array([-1.41303710e+00, -8.01699935e-04, -4.41559986e-03])

In [129]:
lifu_on_time

array([1606261.3700451, 1606268.3756182, 1606277.1030768, 1606348.3901033])

In [130]:
target_ts

array([1606269.7886553, 1606277.1038785, 1606348.3945189])